<a href="https://colab.research.google.com/github/Irosha-Ranaweera/ai-image-detection-generalization/blob/main/notebooks/02_colab_full_experiment_baseline_vs_eca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 02 - Full Experiment in Colab

This notebook runs the full comparison for AI-generated image detection:

1. Baseline ResNet18
2. ResNet18 + Efficient Channel Attention (ECA)

The dataset must have this structure:

```text
ddata/
  train/
    fake/
    real/
  val/
    fake/
    real/
  test/
    fake/
    real/
```

The metrics in this project treat `fake` as the positive class.

## 1. Select GPU

In Colab, choose:

`Runtime > Change runtime type > GPU`

Then run the cell below.

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU. Change runtime type to GPU before full training.")

CUDA available: True
GPU: Tesla T4


## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Go to the project folder

Update `PROJECT_DIR` if your repository is stored in a different Google Drive folder.

In [3]:
PROJECT_DIR = "/content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/ai-image-detection-generalization"

%cd "$PROJECT_DIR"
!pwd
!ls

/content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/ai-image-detection-generalization
/content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/ai-image-detection-generalization
01_test_setup.ipynb    configs	  outputs    report	       scripts
01_vastai_setup.ipynb  notebooks  README.md  requirements.txt  src


## 4. Install requirements

In [4]:
!pip install -q torch torchvision scikit-learn matplotlib tqdm seaborn

## 5. Set full dataset path

Your Windows path:

```text
G:\My Drive\Master of AI ML\Semester01-2026\Deep learning\Deep learning Assignment\datasets\ddata
```

becomes this Colab path:

```text
/content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/datasets/ddata
```

In [5]:
DATA_DIR = "/content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/datasets/ddata"

!ls "$DATA_DIR"
!ls "$DATA_DIR/train"
!ls "$DATA_DIR/val"
!ls "$DATA_DIR/test"

test  train  val
fake  real
fake  real
fake  real


## 6. Count dataset images

This confirms the number of images in each split and class before training.

In [ ]:
from pathlib import Path

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
data_path = Path(DATA_DIR)

for split in ["train", "val", "test"]:
    for class_name in ["fake", "real"]:
        folder = data_path / split / class_name
        count = sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in image_extensions)
        print(f"{split}/{class_name}: {count}")

## 7. Common experiment settings

Use the same settings for both models so the comparison is fair.

In [6]:
import os

os.environ["DATA_DIR"] = DATA_DIR
os.environ["MODEL_NAME"] = "resnet18"
os.environ["EPOCHS"] = "10"
os.environ["BATCH_SIZE"] = "32"
os.environ["LEARNING_RATE"] = "1e-4"
os.environ["NUM_WORKERS"] = "2"
os.environ["OUTPUT_DIR"] = "outputs/figures"

print("DATA_DIR:", os.environ["DATA_DIR"])
print("MODEL_NAME:", os.environ["MODEL_NAME"])
print("EPOCHS:", os.environ["EPOCHS"])
print("BATCH_SIZE:", os.environ["BATCH_SIZE"])
print("LEARNING_RATE:", os.environ["LEARNING_RATE"])
print("NUM_WORKERS:", os.environ["NUM_WORKERS"])
print("OUTPUT_DIR:", os.environ["OUTPUT_DIR"])

DATA_DIR: /content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/datasets/ddata
MODEL_NAME: resnet18
EPOCHS: 10
BATCH_SIZE: 32
LEARNING_RATE: 1e-4
NUM_WORKERS: 2
OUTPUT_DIR: outputs/figures


## 8. Train baseline ResNet18

This trains the baseline model and evaluates it on the test set.

Saved model:

```text
best_resnet18.pth
```

In [10]:
!git status


Refresh index: 100% (37/37), done.
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [12]:
!ls
!git pull

01_test_setup.ipynb    configs	  outputs    report	       scripts
01_vastai_setup.ipynb  notebooks  README.md  requirements.txt  src
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 20 (delta 15), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (20/20), 4.14 KiB | 6.00 KiB/s, done.
From https://github.com/Irosha-Ranaweera/ai-image-detection-generalization
   ed04a3f..80d1109  main       -> origin/main
Updating ed04a3f..80d1109
Fast-forward
 .../02_colab_full_experiment_baseline_vs_eca.ipynb | 775 +++++++++++++--------
 scripts/create_sample_dataset.py                   |  37 +-
 2 files changed, 504 insertions(+), 308 deletions(-)


In [6]:
SOURCE_DIR = "/content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/datasets/ddata"

SUBSET_DIR = "/content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/datasets/ddata_subset_3000_750_750"


In [8]:
!python scripts/create_sample_dataset.py \
  --source "$SOURCE_DIR" \
  --output "$SUBSET_DIR" \
  --train-per-class 3000 \
  --val-per-class 750 \
  --test-per-class 750 \
  --seed 42


train/real: copied 3000 images
train/fake: copied 3000 images
val/real: copied 750 images
val/fake: copied 750 images
test/real: copied 750 images
test/fake: copied 750 images

Done. Copied 9000 images to: /content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/datasets/ddata_subset_3000_750_750


In [9]:
ZIP_PATH = "/content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/datasets/ddata_subset_3000_750_750.zip"

!cd "$(dirname "$SUBSET_DIR")" && zip -r "$ZIP_PATH" "$(basename "$SUBSET_DIR")"


Streaming output truncated to the last 5000 lines.
  adding: ddata_subset_3000_750_750/train/fake/fake_033772.jpg (deflated 1%)
  adding: ddata_subset_3000_750_750/train/fake/fake_023628.jpg (deflated 1%)
  adding: ddata_subset_3000_750_750/train/fake/fake_009627.jpg (deflated 1%)
  adding: ddata_subset_3000_750_750/train/fake/fake_022220.jpg (deflated 1%)
  adding: ddata_subset_3000_750_750/train/fake/fake_018855.jpg (deflated 1%)
  adding: ddata_subset_3000_750_750/train/fake/fake_036725.jpg (deflated 1%)
  adding: ddata_subset_3000_750_750/train/fake/fake_009026.jpg (deflated 1%)
  adding: ddata_subset_3000_750_750/train/fake/fake_009514.jpg (deflated 1%)
  adding: ddata_subset_3000_750_750/train/fake/fake_006894.jpg (deflated 1%)
  adding: ddata_subset_3000_750_750/train/fake/fake_015881.jpg (deflated 1%)
  adding: ddata_subset_3000_750_750/train/fake/fake_037292.jpg (deflated 1%)
  adding: ddata_subset_3000_750_750/train/fake/fake_041406.jpg (deflated 1%)
  adding: ddata_subset_30

In [10]:
# Dataset copied locally for speed
!cp -r "$SUBSET_DIR" /content/ddata_subset_3000_750_750
DATA_DIR = "/content/ddata_subset_3000_750_750"




In [14]:
!ls "$DATA_DIR"
!ls "$DATA_DIR/train"
!ls "$DATA_DIR/val"
!ls "$DATA_DIR/test"


test  train  val
fake  real
fake  real
fake  real


In [15]:
import os

os.environ.pop("CHECKPOINT_PATH", None)
os.environ["EPOCHS"] = "3"
DATA_DIR = "/content/ddata_subset_3000_750_750"
os.environ["DATA_DIR"] = DATA_DIR
# os.environ["SAVE_PATH"] = "outputs/checkpoints/best_resnet18.pth"
RESULTS_DIR = "/content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/results"

# Results saved to Drive for safety
os.environ["SAVE_PATH"] = f"{RESULTS_DIR}/checkpoints/best_resnet18.pth"
os.environ["OUTPUT_DIR"] = f"{RESULTS_DIR}/figures"


!python scripts/train_baseline.py

Using device: cpu
Classes: ['fake', 'real']
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 105MB/s]

Epoch 1/3
Train Loss: 0.6842 | Train Acc: 0.5780
Val Loss:   0.6251 | Val Acc:   0.6667
Saved new best model to /content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/results/checkpoints/best_resnet18.pth

Epoch 2/3
Train Loss: 0.6079 | Train Acc: 0.6752
Val Loss:   0.5781 | Val Acc:   0.6900
Saved new best model to /content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/results/checkpoints/best_resnet18.pth

Epoch 3/3
Train Loss: 0.5717 | Train Acc: 0.7128
Val Loss:   0.5493 | Val Acc:   0.7180
Saved new best model to /content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/results/checkpoints/best_resnet18.pth

Best validation accuracy: 0.7180

Test Results


### Run from checkpoint

In [ ]:
os.environ["CHECKPOINT_PATH"] =f"{RESULTS_DIR}/checkpoints/best_resnet18.pth"
os.environ["SAVE_PATH"] =f"{RESULTS_DIR}/checkpoints/best_resnet18_001.pth"
os.environ["EPOCHS"] = "7"

!python scripts/train_baseline.py


Using device: cpu
Classes: ['fake', 'real']
Loaded checkpoint: /content/drive/MyDrive/Master of AI ML/Semester01-2026/Deep learning/Deep learning Assignment/results/checkpoints/best_resnet18.pth

Epoch 1/7
Training:   6% 12/188 [00:47<11:21,  3.87s/it]

## 9. Train ResNet18 + ECA

This trains the attention-enhanced model using the same dataset and hyperparameters.

Saved model:

```text
best_eca_resnet18.pth
```

In [16]:
os.environ.pop("CHECKPOINT_PATH", None)
os.environ["SAVE_PATH"] = "outputs/checkpoints/best_eca_resnet18.pth"

!python scripts/train_attention.py

Using device: cpu
Classes: ['fake', 'real']

Epoch 1/3
Train Loss: 0.6724 | Train Acc: 0.5890
Val Loss:   0.6461 | Val Acc:   0.6467
Saved new best model to outputs/checkpoints/best_eca_resnet18.pth

Epoch 2/3
Train Loss: 0.6229 | Train Acc: 0.6730
Val Loss:   0.6070 | Val Acc:   0.6827
Saved new best model to outputs/checkpoints/best_eca_resnet18.pth

Epoch 3/3
Train Loss: 0.5864 | Train Acc: 0.7128
Val Loss:   0.5767 | Val Acc:   0.7093
Saved new best model to outputs/checkpoints/best_eca_resnet18.pth

Best validation accuracy: 0.7093

Test Results
Positive class: fake (label 0)
Accuracy : 0.7373
Precision: 0.7124
Recall   : 0.796
F1 Score : 0.7519
ROC-AUC  : 0.8174

Classification Report:

              precision    recall  f1-score   support

        fake       0.71      0.80      0.75       750
        real       0.77      0.68      0.72       750

    accuracy                           0.74      1500
   macro avg       0.74      0.74      0.74      1500
weighted avg       0.74   

## Experiment Summary - 1st step

dataset size:

| Split | Fake | Real | Total |
|---|---:|---:|---:|
| Train | 3000 | 3000 | 6000 |
| Validation | 750 | 750 | 1500 |
| Test | 750 | 750 | 1500 |
| **Total** | **4500** | **4500** | **9000** |

 **3 epochs** on **CPU**

| Model | Accuracy | Precision(fake) | Recall(fake) | F1(fake) | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| ResNet18 baseline | 0.7967 | 0.8654 | 0.7027 | 0.7756 | 0.8843 |
| ResNet18 + ECA | 0.7373 | 0.7124 | 0.7960 | 0.7519 | 0.8174 |

The baseline ResNet18 model achieved higher overall accuracy, precision, F1-score, and ROC-AUC. However, the ResNet18 + ECA model achieved higher recall for the fake class, meaning it detected more AI-generated images but also produced more false positives on real images.


## 11. Interpretation guide

- `Accuracy`: total correct predictions across fake and real images.
- `Precision(fake)`: when the model predicts fake, how often it is correct.
- `Recall(fake)`: how many fake images the model successfully detects.
- `F1(fake)`: balance between fake precision and fake recall.

For this assignment, `Recall(fake)` and `F1(fake)` are especially important because the task is AI-generated image detection.